# nuPlan Paper Experiments

这个 notebook 专门服务论文产出，不负责跑底层 benchmark。

结构按三块组织：

1. `Zero-shot / Transfer`
2. `Fine-tune / In-domain`
3. `Ablation`

论文表格和图统一使用你在 2car validation 里定下来的 5 个指标定义：

- `Safety`: `1 - collision_rate`，对应 2car 里的 `min_dist >= R_COLLISION`
- `Goal`: `goal_reached_rate`
- `TimeToGoal`: `time_to_goal_s`
- `MinDist`: `min_inter_agent_dist_m`
- `RoadViol`: `road_boundary_violations`

注意：2car notebook 只作为 metric 定义参考，不作为 nuPlan map 上的实验结果。nuPlan paper 表必须来自 nuPlan benchmark 在 map/scenario 上跑出来的 `metrics.csv`。

数据来源：

1. `notebooks/cache/nuplan_transfer/metrics.csv`：zero-shot on nuPlan map/scenarios
2. `notebooks/cache/nuplan_transfer_finetuned/metrics.csv`：fine-tune / in-domain
3. external baseline CSVs
4. `abl_res.pkl` 仅用于 debug，对论文 nuPlan 表默认关闭

建议论文写法：

- 把 `Training regime` 明确标成 `zero_shot / fine_tuned / in_domain / no_learning`
- `DiffusionPlanner / FlowPlanner` 当前默认视为 `in_domain` baseline
- zero-shot 表的重点是迁移性和安全性，不是绝对公平的 fully-trained SOTA 对比


In [ ]:
from __future__ import annotations

from pathlib import Path
import pickle
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path('/new_world/cockatiel/ra').resolve()
PROJECT_ROOT = ROOT / 'safeflow-nuplan-transfer' / 'dmpc_fm_cbf'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dmpc_fm_cbf.paper_experiment_reporting import (
    add_paper_metric_aliases,
    attach_method_metadata,
    build_finetune_table,
    build_method_metadata,
    build_paper_metric_table,
    build_zero_shot_table,
    export_latex_table,
    export_markdown_table,
    load_external_baseline_csv,
    load_metrics_csv,
    make_figure_paths,
    merge_metrics_tables,
    plot_ablation,
    plot_regime_comparison,
    plot_zero_shot_comparison,
    summarize_metrics,
)

PAPER_OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'cache' / 'paper_experiments'
PAPER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_PATHS = make_figure_paths(PAPER_OUTPUT_DIR)
PAPER_OUTPUT_DIR


In [ ]:
# ------------------------------
# Result file registry
# ------------------------------
# 你后面只需要改这些路径。

ZERO_SHOT_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'nuplan_transfer' / 'metrics.csv'
FINE_TUNE_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'nuplan_transfer_finetuned' / 'metrics.csv'

# Debug only: 2car validation 不是 nuPlan map 结果，默认不能混进 paper 表。
USE_TWO_CAR_DEBUG_FALLBACK = False
TWO_CAR_ABLATION_RESULTS = PROJECT_ROOT / 'notebooks' / 'cache' / 'abl_res.pkl'
TWO_CAR_DT = 0.1
TWO_CAR_COLLISION_RADIUS_M = 0.20

# 外部 baseline CSV：不存在也没关系，notebook 会跳过。
OFFICIAL_IDM_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'external_baselines' / 'official_idm_metrics.csv'
OFFICIAL_PDM_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'external_baselines' / 'official_pdm_closed_metrics.csv'
DIFFUSION_PLANNER_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'external_baselines' / 'diffusion_planner_metrics.csv'
FLOW_PLANNER_METRICS = PROJECT_ROOT / 'notebooks' / 'cache' / 'external_baselines' / 'flow_planner_metrics.csv'

# 消融 CSV：推荐你后面按这个 schema 保存。
# required columns: paper_bucket, tau0 or K or road_margin_m,
# safety_rate, goal_success_rate, time_to_goal_s,
# min_inter_agent_dist_m, road_boundary_violations
# 兼容旧字段：collision_rate -> safety_rate, goal_reached_rate -> goal_success_rate, min_dist -> min_inter_agent_dist_m。
ABLATION_TAU_CSV = PROJECT_ROOT / 'notebooks' / 'cache' / 'paper_experiments' / 'ablation_tau.csv'
ABLATION_K_CSV = PROJECT_ROOT / 'notebooks' / 'cache' / 'paper_experiments' / 'ablation_k.csv'
ABLATION_MARGIN_CSV = PROJECT_ROOT / 'notebooks' / 'cache' / 'paper_experiments' / 'ablation_margin.csv'

RESULT_PATHS = {
    'zero_shot': ZERO_SHOT_METRICS,
    'fine_tune': FINE_TUNE_METRICS,
    'two_car_debug_fallback': TWO_CAR_ABLATION_RESULTS,
    'official_idm': OFFICIAL_IDM_METRICS,
    'official_pdm_closed': OFFICIAL_PDM_METRICS,
    'diffusion_planner': DIFFUSION_PLANNER_METRICS,
    'flow_planner': FLOW_PLANNER_METRICS,
}
RESULT_PATHS


In [ ]:
def _install_numpy_pickle_compat() -> None:
    import importlib
    import sys

    aliases = {
        'numpy._core': 'numpy.core',
        'numpy._core.multiarray': 'numpy.core.multiarray',
        'numpy._core.numeric': 'numpy.core.numeric',
        'numpy._core.umath': 'numpy.core.umath',
        'numpy._core._multiarray_umath': 'numpy.core._multiarray_umath',
    }
    for missing_name, fallback_name in aliases.items():
        if missing_name in sys.modules:
            continue
        try:
            sys.modules[missing_name] = importlib.import_module(fallback_name)
        except ModuleNotFoundError:
            pass


class _NumpyCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith('numpy._core'):
            module = module.replace('numpy._core', 'numpy.core', 1)
        return super().find_class(module, name)


def _load_pickle_with_numpy_compat(path: Path):
    _install_numpy_pickle_compat()
    with path.open('rb') as f:
        try:
            return pickle.load(f)
        except ModuleNotFoundError as exc:
            if exc.name and exc.name.startswith('numpy._core'):
                f.seek(0)
                return _NumpyCompatUnpickler(f).load()
            raise


def load_two_car_ablation_results(
    path: Path,
    *,
    dt: float = TWO_CAR_DT,
    collision_radius_m: float = TWO_CAR_COLLISION_RADIUS_M,
) -> pd.DataFrame:
    results = _load_pickle_with_numpy_compat(path)

    rows = []
    for method, trials in results.items():
        for r in trials:
            min_dist = float(r.get('min_dist', np.nan))
            collision = bool(r.get('collision', min_dist < collision_radius_m))
            success_a = bool(r.get('success_A', False))
            success_b = bool(r.get('success_B', False))
            scenario = r.get('scenario') or 'validation'

            final_dists = []
            if 'trajA' in r and 'gA' in r:
                final_dists.append(float(np.linalg.norm(np.asarray(r['trajA'])[-1, :2] - np.asarray(r['gA']))))
            if 'trajB' in r and 'gB' in r:
                final_dists.append(float(np.linalg.norm(np.asarray(r['trajB'])[-1, :2] - np.asarray(r['gB']))))

            rows.append({
                'method': method,
                'paper_bucket': f'2car_{scenario}',
                'time_to_goal_s': float(r.get('steps', np.nan)) * dt,
                'collision': collision,
                'goal_reached': success_a and success_b,
                'min_inter_agent_dist_m': min_dist,
                'road_boundary_violations': float(r.get('road_boundary_violations', 0.0)),
                'final_dist_to_goal_m': float(np.mean(final_dists)) if final_dists else np.nan,
            })

    if not rows:
        raise RuntimeError(f'No trials found in fallback result file: {path}')
    return pd.DataFrame(rows)


tables = []

if ZERO_SHOT_METRICS.exists():
    zero_df = load_metrics_csv(ZERO_SHOT_METRICS)
    tables.append(zero_df)
    print('[OK] zero-shot metrics:', ZERO_SHOT_METRICS)
else:
    print('[MISS] zero-shot metrics:', ZERO_SHOT_METRICS)

if FINE_TUNE_METRICS.exists():
    ft_df = load_metrics_csv(
        FINE_TUNE_METRICS,
        method_overrides={
            'fm_cbf': 'fm_cbf_finetuned',
            'pure_fm': 'pure_fm_finetuned',
        },
    )
    tables.append(ft_df)
    print('[OK] fine-tune metrics:', FINE_TUNE_METRICS)
else:
    print('[MISS] fine-tune metrics:', FINE_TUNE_METRICS)

for method_key, path in [
    ('official_idm', OFFICIAL_IDM_METRICS),
    ('official_pdm_closed', OFFICIAL_PDM_METRICS),
    ('diffusion_planner', DIFFUSION_PLANNER_METRICS),
    ('flow_planner', FLOW_PLANNER_METRICS),
]:
    if path.exists():
        ext_df = load_external_baseline_csv(path)
        if 'method' not in ext_df.columns or ext_df['method'].nunique() != 1:
            ext_df['method'] = method_key
        tables.append(ext_df)
        print(f'[OK] external baseline {method_key}:', path)
    else:
        print(f'[MISS] external baseline {method_key}:', path)

if not tables and USE_TWO_CAR_DEBUG_FALLBACK and TWO_CAR_ABLATION_RESULTS.exists():
    two_car_df = load_two_car_ablation_results(TWO_CAR_ABLATION_RESULTS)
    tables.append(two_car_df)
    fallback_csv = PAPER_OUTPUT_DIR / 'two_car_debug_fallback_metrics.csv'
    two_car_df.to_csv(fallback_csv, index=False)
    print('[DEBUG OK] 2car fallback metrics:', TWO_CAR_ABLATION_RESULTS)
    print('saved:', fallback_csv)
elif not tables and USE_TWO_CAR_DEBUG_FALLBACK:
    print('[MISS] 2car debug fallback metrics:', TWO_CAR_ABLATION_RESULTS)
elif not tables:
    print('[INFO] 2car debug fallback is disabled because it is not a nuPlan map result.')

if not tables:
    raise RuntimeError(
        'No nuPlan metrics found. Run zero-shot nuPlan benchmark first so '
        f'{ZERO_SHOT_METRICS} exists, or place external baseline CSVs in the configured paths.'
    )

metrics_df = merge_metrics_tables(tables)
method_meta = build_method_metadata(metrics_df['method'].unique())
metrics_enriched = attach_method_metadata(metrics_df, method_meta)
summary_df = summarize_metrics(metrics_enriched)
summary_table = build_paper_metric_table(summary_df)

summary_csv = PAPER_OUTPUT_DIR / 'summary_metrics.csv'
summary_df.to_csv(summary_csv, index=False)
summary_table.to_csv(PAPER_OUTPUT_DIR / 'summary_paper_metrics.csv', index=False)
print('saved:', summary_csv)
print('saved:', PAPER_OUTPUT_DIR / 'summary_paper_metrics.csv')
display(summary_table.head(20))


## Zero-shot Table

这张表建议放：

- `SafeFlow (FM+CBF)`
- `Pure FM`
- `IDM / PDM-Closed`
- `DiffusionPlanner / FlowPlanner`

但要保留 `training_regime` 一列。

In [ ]:
from IPython.display import Image, display
from dmpc_fm_cbf.paper_experiment_reporting import (
    build_zero_shot_table,
    export_latex_table,
    export_markdown_table,
    plot_zero_shot_comparison,
)

zero_shot_table = build_zero_shot_table(summary_df)
display(zero_shot_table)

export_markdown_table(zero_shot_table, PAPER_OUTPUT_DIR / 'zero_shot_table.md')
export_latex_table(zero_shot_table, PAPER_OUTPUT_DIR / 'zero_shot_table.tex')
plot_zero_shot_comparison(summary_df, FIG_PATHS.zero_shot_barplot)
display(Image(filename=str(FIG_PATHS.zero_shot_barplot)))


## Fine-tune Table

这张表建议对比：

- `SafeFlow (Fine-tuned)`
- `Pure FM (Fine-tuned)`
- `IDM / PDM-Closed`
- `DiffusionPlanner / FlowPlanner`

如果你还没有 fine-tune 结果，这一节会先退化成基线占位。

In [ ]:
from IPython.display import Image, display
from dmpc_fm_cbf.paper_experiment_reporting import (
    build_finetune_table,
    export_latex_table,
    export_markdown_table,
    plot_regime_comparison,
)

fine_tune_table = build_finetune_table(summary_df)
display(fine_tune_table)

export_markdown_table(fine_tune_table, PAPER_OUTPUT_DIR / 'fine_tune_table.md')
export_latex_table(fine_tune_table, PAPER_OUTPUT_DIR / 'fine_tune_table.tex')
plot_regime_comparison(summary_df, FIG_PATHS.regime_barplot)
display(Image(filename=str(FIG_PATHS.regime_barplot)))


## Ablation

推荐三组消融：

- `tau0` / `tau1` 门控
- `K` 候选轨迹数
- `road_margin_m`

CSV schema 建议：

- `paper_bucket`
- `tau0` 或 `K` 或 `road_margin_m`
- `safety_rate` (`Safety`)
- `goal_success_rate` (`Goal`)
- `time_to_goal_s` (`TimeToGoal`)
- `min_inter_agent_dist_m` (`MinDist`)
- `road_boundary_violations` (`RoadViol`)

兼容旧字段：`collision_rate`、`goal_reached_rate`、`min_dist` 会在下面自动映射。


In [ ]:
def normalize_paper_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = add_paper_metric_aliases(df)
    if 'min_inter_agent_dist_m' not in out.columns and 'min_dist' in out.columns:
        out['min_inter_agent_dist_m'] = out['min_dist']
    if 'road_boundary_violations' not in out.columns:
        out['road_boundary_violations'] = 0.0
    return out

PAPER_ABLATION_METRICS = [
    ('safety_rate', 'Safety ↑'),
    ('goal_success_rate', 'Goal ↑'),
    ('time_to_goal_s', 'TimeToGoal ↓'),
    ('min_inter_agent_dist_m', 'MinDist ↑'),
    ('road_boundary_violations', 'RoadViol ↓'),
]

if ABLATION_TAU_CSV.exists():
    tau_df = normalize_paper_metrics(pd.read_csv(ABLATION_TAU_CSV))
    display(tau_df)
    plot_ablation(
        tau_df,
        x_col='tau0',
        metrics=PAPER_ABLATION_METRICS,
        title='CBF Gate Ablation',
        output_path=FIG_PATHS.ablation_tau,
    )
    display(Image(filename=str(FIG_PATHS.ablation_tau)))
else:
    print('[MISS] tau ablation csv:', ABLATION_TAU_CSV)

if ABLATION_K_CSV.exists():
    k_df = normalize_paper_metrics(pd.read_csv(ABLATION_K_CSV))
    display(k_df)
    plot_ablation(
        k_df,
        x_col='K',
        metrics=PAPER_ABLATION_METRICS,
        title='Candidate Count Ablation',
        output_path=FIG_PATHS.ablation_k,
    )
    display(Image(filename=str(FIG_PATHS.ablation_k)))
else:
    print('[MISS] K ablation csv:', ABLATION_K_CSV)

if ABLATION_MARGIN_CSV.exists():
    margin_df = normalize_paper_metrics(pd.read_csv(ABLATION_MARGIN_CSV))
    display(margin_df)
    plot_ablation(
        margin_df,
        x_col='road_margin_m',
        metrics=PAPER_ABLATION_METRICS,
        title='Road CBF Margin Ablation',
        output_path=FIG_PATHS.ablation_margin,
    )
    display(Image(filename=str(FIG_PATHS.ablation_margin)))
else:
    print('[MISS] road margin ablation csv:', ABLATION_MARGIN_CSV)


## Files Produced

这个 notebook 会输出：

- `summary_metrics.csv`：保留原始字段和论文指标别名
- `summary_paper_metrics.csv`：只保留论文展示指标 `Safety / Goal / TimeToGoal / MinDist / RoadViol`
- `zero_shot_table.md/.tex`
- `fine_tune_table.md/.tex`
- `zero_shot_comparison.png`
- `regime_comparison.png`
- `ablation_tau.png`
- `ablation_k.png`
- `ablation_margin.png`

注意：这些文件必须来自 nuPlan benchmark 的 `metrics.csv`。`two_car_debug_fallback_metrics.csv` 只在 `USE_TWO_CAR_DEBUG_FALLBACK = True` 时用于调试，不能当作 nuPlan paper 结果。

都在：

- `/new_world/cockatiel/ra/safeflow-nuplan-transfer/dmpc_fm_cbf/notebooks/cache/paper_experiments/`
